## Setup

In [37]:
import serial
import csv
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
from scipy.special import gammaincc
from scipy.stats import norm
import numpy.fft as fft

import pyqtgraph as pg
from pyqtgraph.Qt import QtCore
from scipy.signal import welch

PORT = "COM8"
BAUD = 230400

LOAD_FROM_FILE = False
LOAD_FILE_PATH = Path("data/sample2.csv")

SENSORS = ['BPW34_1', 'BPW34_2']

ADC_MAX = 16383 #2^14 - 1
V_REF = 5 #confirmed

SAMPLES_PER_PACKET = 50
payload_dtype = np.dtype([
    ('timestamp', '<u4'),
    ('BPW34_1', '<u2'),
    ('BPW34_2', '<u2'),
    ('Inano_1', '<u2'),
    ('Inano_2', '<u2')
])

BYTES_PER_SAMPLE = payload_dtype.itemsize * SAMPLES_PER_PACKET
CARRIER_FREQ = 500  # carrier frequency in Hz
ser = None
reader = None

def fir_lowpass_filter(data, cutoff_hz, fs, num_taps=101):
    b = signal.firwin(num_taps, cutoff=cutoff_hz, fs=fs, window='hamming')
    filtered_data = np.asarray(signal.lfilter(b, 1.0, data))
    return filtered_data, b

def adc_to_mv(adc_value):
    return (adc_value / ADC_MAX) * V_REF * 1000  # Convert to millivolts

def num_samples_in_file(file_path):
    df = pd.read_csv(file_path)
    times = df['timestamp'].to_numpy()
    return len(times)

def fmt_matrix(M, title, fmt="{:>12.4f}"):
    s = f"\n{title}\n"
    s += " " * 10 + "".join(f"{ch:>12}" for ch in SENSORS) + "\n"
    for i, ch in enumerate(SENSORS):
        s += f"{ch:>10}" + "".join(fmt.format(M[i, j]) for j in range(len(SENSORS))) + "\n"
    return s

# reads 50 samples from serial port.
def try_read_realtime(ser):
    if ser.read(1) == b'\xAA' and ser.read(1) == b'\xBB':
        raw_data = ser.read(BYTES_PER_SAMPLE) 
        if len(raw_data) == BYTES_PER_SAMPLE:
            data = np.frombuffer(raw_data, dtype=payload_dtype)
            return data
    return None

# read 50 samples from file to emulate realtime reading.
def read_from_file_realtime(file_path):
    df = pd.read_csv(file_path)
    data = df[list(payload_dtype.names or ())].to_records(index=False).astype(payload_dtype)
    n_packets = len(data) // SAMPLES_PER_PACKET
    packets = [data[i * SAMPLES_PER_PACKET:(i + 1) * SAMPLES_PER_PACKET]
               for i in range(n_packets)]

    idx = 0
    ts_start = int(packets[0]['timestamp'][0]) if packets else 0  # first timestamp (us)
    wall_start = None  # wall-clock time

    def reader(*_): 
        nonlocal idx, wall_start
        if idx >= len(packets):
            return np.zeros(SAMPLES_PER_PACKET, dtype=payload_dtype)  # exhausted
        pkt = packets[idx]
        now = time.time()
        if wall_start is None:  # start the clock on the first call
            wall_start = now
        # only release the packet once real time has caught up to its timestamp
        elapsed_us = (now - wall_start) * 1e6
        if elapsed_us < int(pkt['timestamp'][0]) - ts_start:
            return None
        idx += 1
        return pkt

    return reader

def read_from_file_instant(file_path):
    df = pd.read_csv(file_path)
    data = df.to_records(index=False).astype(payload_dtype)
    return data


def save_data(samples_total, savefile_path, ser):
    savefile_path.parent.mkdir(parents=True, exist_ok=True)

    header_written = False
    sample_count = 0

    ##read packets of data from the serial port and save it to a CSV file
    try:
        while samples_total is None or sample_count < samples_total:
            data = try_read_realtime(ser)
            if data is not None:
                mean_delta_us = np.mean(np.diff(data['timestamp']))
                if mean_delta_us > 0:
                    # write all samples in this packet to file
                    df = pd.DataFrame(data)
                    df.to_csv(savefile_path, mode='a', header=not header_written, index=False)
                    header_written = True
                    sample_count += SAMPLES_PER_PACKET
                else:
                    print("Timestamps did not change")
    except KeyboardInterrupt:
        print("Stopping...")
    finally:
        print(f"Saved {sample_count} samples to {savefile_path.as_posix()}")





start = time.time()
time_chunks = []
full_data = {s: [] for s in SENSORS}
Fs = 1500.0  # initial guess for sampling rate
ser = serial.Serial(PORT, BAUD, timeout=1) if not LOAD_FROM_FILE else None

if LOAD_FROM_FILE:
    reader = read_from_file_realtime(LOAD_FILE_PATH)

while time.time() - start < 5:
    if LOAD_FROM_FILE:
        assert reader is not None 
        data = reader()
    else:
        data = try_read_realtime(ser)
    if data is not None:
        time_chunks.append(data['timestamp'])
        for s in SENSORS:
            full_data[s].append(adc_to_mv(data[s]))
if ser is not None and ser.is_open:
    ser.close()

Fs = float(np.mean([1.0 / (np.median(np.diff(time_chunks[i])) * 1e-6) for i in range(len(time_chunks))]))
print(f"Estimated sampling rate: {Fs:.2f} Hz")

estimated_carrier_freqs = {}

for s in SENSORS:
    x = np.concatenate(full_data[s])  # full_data[s] is a list of per-packet chunks
    f, Pxx = welch(x - np.mean(x), fs=Fs, nperseg=2048)
    inband = f > 80  # skip the DC / drift / 1-f region
    estimate_carrier_freq = f[inband][np.argmax(Pxx[inband])]
    estimated_carrier_freqs[s] = estimate_carrier_freq
    print(f"{s}: estimated carrier frequency = {estimate_carrier_freq:.2f} Hz")

CARRIER_FREQ = np.mean(list(estimated_carrier_freqs.values()))
print(f"Average estimated carrier frequency across sensors: {CARRIER_FREQ:.2f} Hz")

Estimated sampling rate: 1499.25 Hz
BPW34_1: estimated carrier frequency = 499.26 Hz
BPW34_2: estimated carrier frequency = 499.26 Hz
Average estimated carrier frequency across sensors: 499.26 Hz


## Saves realtime data to file


In [ ]:
ser = serial.Serial(PORT, BAUD, timeout=1)
NUM_SAMPLES_TO_SAVE = 10000   # set to None to run until interrupted

now = datetime.now()
savefile_path = Path(f"data/{now.strftime('%m_%d_%Y')}/run_{now.strftime('%H_%M_%S')}.csv")

save_data(NUM_SAMPLES_TO_SAVE, savefile_path, ser)

if ser is not None and ser.is_open:
    ser.close()

## Realtime plot

Reads packets from the Arduino and draws them sweep-style (like an ECG monitor): each incoming 50-sample packet is written at the cursor — the blank gap in the trace — and everything already drawn stays put until the cursor wraps around and overwrites it.

In [45]:

if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)
    ser.reset_input_buffer()  # drop bytes buffered before this cell started so we begin on live data


N = 10000
assert N % SAMPLES_PER_PACKET == 0
buffers = {s: np.full(N, np.nan) for s in SENSORS}  # nan = not drawn (unfilled region / cursor gap)
write_idx = 0

FILTER = True

label_style = {'color': '#FFF', 'font-size': '16pt'}

app = pg.mkQApp("realtime")
win = pg.GraphicsLayoutWidget(show=True, title="realtime")
pen = pg.mkPen('r', width=5)
curves = {}
for i, s in enumerate(SENSORS):
    p = win.ci.addPlot(row=i, col=0)
    p.setLabel('left', 'mV', **label_style)
    p.setLabel('top', f'{s}', **label_style)
    curves[s] = p.plot(pen=pen, connect='finite')  # 'finite' leaves a gap at the nan cursor instead of a line across it

sos_bp = signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos')
sos_zi = {s: signal.sosfilt_zi(sos_bp) for s in SENSORS}

if LOAD_FROM_FILE:
    reader = read_from_file_realtime(LOAD_FILE_PATH)

def update():
    # Sweep display: each new packet overwrites the 50 samples at the cursor and
    # nothing else moves, so previously plotted values never change.
    global write_idx
    got_packet = False

    while True:
        if LOAD_FROM_FILE:
            data = reader()
            if data is None:  # reader is pacing itself against wall clock
                break
        else:
            if ser.in_waiting < BYTES_PER_SAMPLE + 2:
                break  # no complete packet buffered; never block the GUI thread
            data = try_read_realtime(ser)
            if data is None:
                continue  # header scan slipped a byte or two; keep scanning the backlog
        for s in SENSORS:
            if FILTER:
                cleaned, sos_zi[s] = signal.sosfilt(sos_bp, data[s], zi=sos_zi[s])
                buffers[s][write_idx:write_idx + SAMPLES_PER_PACKET] = adc_to_mv(cleaned)
            else:
                buffers[s][write_idx:write_idx + SAMPLES_PER_PACKET] = adc_to_mv(data[s])
        write_idx = (write_idx + SAMPLES_PER_PACKET) % N
        got_packet = True
        if LOAD_FROM_FILE:
            break  # one packet per tick; the file reader releases them at the recorded pace
    if got_packet:
        for s in SENSORS:
            buffers[s][write_idx:write_idx + SAMPLES_PER_PACKET] = np.nan  # blank slot marks the cursor
            curves[s].setData(buffers[s])

timer = QtCore.QTimer()
timer.timeout.connect(update)
timer.start(10)
pg.exec()
timer.stop()  # don't leave a stale timer attached if the cell is re-run
if ser is not None and ser.is_open:
    ser.close()

## Spectrogram Plotting

Make sure to specify load_from_file in the setup block to load from file instead of read real time data

In [ ]:
from scipy.signal import get_window


if not LOAD_FROM_FILE:  
    ser = serial.Serial(PORT, BAUD, timeout=1)
NFFT = 4096        
WINDOWSIZE = 2048
WINDOW = 'hamming'
OVERLAP = 90 
FILTER = False
CAPTURE_TIME = 3  # seconds

## load from file or read from serial
## after this block: samples[ch] and times are plain 1-D numpy arrays
if LOAD_FROM_FILE:
    reader = read_from_file_realtime(LOAD_FILE_PATH)
    

sample_chunks = {s: [] for s in SENSORS}
time_chunks = []
start = time.time()

# !!! ENSURE DATA IN FILE IS ATLEAST AS LONG AS CAPTURE TIME
while time.time() - start < CAPTURE_TIME:
    if LOAD_FROM_FILE:
        data = reader()
    else:
        data = try_read_realtime(ser)
    if data is not None:
        time_chunks.append(data['timestamp'])
        for s in SENSORS:
            sample_chunks[s].append(adc_to_mv(data[s]))
if ser is not None and ser.is_open:
    ser.close()

samples = {s: np.concatenate(sample_chunks[s]) for s in SENSORS}
times = np.concatenate(time_chunks)

Fs = 1.0 / (np.median(np.diff(times)) * 1e-6)
win = get_window(WINDOW, WINDOWSIZE)
noverlap = int(WINDOWSIZE * OVERLAP / 100)

sos_bp = signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos') 
filtered = {s: signal.sosfilt(sos_bp, samples[s]) for s in SENSORS}

with plt.style.context('dark_background'):
    fig, axes = plt.subplots(4, 1, figsize=(19, 21), sharex=False, constrained_layout=True)
    fig.suptitle(f'Channel spectrograms, Sampling Rate = {Fs:.2f} Hz', fontsize=30)

    for ax, s in zip(axes, SENSORS):
        if FILTER:
            spec, freqs, t_bins, im = ax.specgram(filtered[s], Fs=Fs, NFFT=WINDOWSIZE, window=win, noverlap=noverlap, pad_to=NFFT, cmap='magma')
        else:
            spec, freqs, t_bins, im = ax.specgram(samples[s], Fs=Fs, NFFT=WINDOWSIZE, window=win, noverlap=noverlap, pad_to=NFFT, cmap='magma')
        ax.set_ylabel(f'{s}\n(Hz)', fontsize = 30)
        ax.set_ylim(0, 600)
        cbar = fig.colorbar(im, ax=ax, pad=0.01)
        cbar.set_label('Power (dB)', fontsize = 30)

    axes[-1].set_xlabel('Time (s)', fontsize = 30)
    plt.show()


## Realtime PSD

In [ ]:

if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)

FILTER = True # set to True to enable bandpass filtering

N = 2000
buffers = {s: np.zeros(N) for s in SENSORS}

# per-channel bandpass filter (29.5 - 30.5 Hz), each keeps its own state
sos_bp = signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos')
sos_zi = {s: signal.sosfilt_zi(sos_bp) for s in SENSORS}

app = pg.mkQApp("QRNG PSD realtime")
win = pg.GraphicsLayoutWidget(show=True, title="QRNG PSD realtime")
curves = {}
plots = []
xticks = [[(1, '1')] + [(f, str(f)) for f in range(100, 1001, 100)]]  # 1, 100, 200, ... 1000
for i, s in enumerate(SENSORS):
    p = win.ci.addPlot(row=0, col=i, title=s)
    p.setLabel('left', 'PSD (mV^2/Hz)')
    p.setLabel('bottom', 'Frequency (Hz)')
    p.getViewBox().setXRange(1, 1000, padding=0)
    p.getViewBox().setYRange(0,2000e-3,padding=0)  # set a minimum y-range to avoid auto-scaling to zero
    p.getAxis('bottom').setTicks(xticks)  # force 1, 100, 200, ... 1000 labels
    if plots:
        p.getViewBox().setYLink(plots[0].getViewBox())  # uniform y axis across all 4 sensors
    plots.append(p)
    curves[s] = p.plot()

if LOAD_FROM_FILE:
    reader = read_from_file_realtime(LOAD_FILE_PATH)

def update():
    if LOAD_FROM_FILE:
        data = reader()
    else:
        data = try_read_realtime(ser)
    if data is not None:
        for s in SENSORS:
            chunk_clean = (data[s] / ADC_MAX * V_REF)*1000 ##from v to mV
            # bandpass this chunk, carrying filter state across chunks
            chunk, sos_zi[s] = signal.sosfilt(sos_bp, chunk_clean, zi=sos_zi[s])
            buffers[s] = np.roll(buffers[s], -SAMPLES_PER_PACKET)
            if FILTER:
                buffers[s][-SAMPLES_PER_PACKET:] = chunk
            else:
                buffers[s][-SAMPLES_PER_PACKET:] = chunk_clean
            f, pxx = welch(buffers[s], fs=Fs, nperseg=512)
            curves[s].setData(f, pxx)

timer = QtCore.QTimer()
timer.timeout.connect(update)
timer.start()
pg.exec()
if ser is not None and ser.is_open:
    ser.close()

## Find the Variance Between Sensors

$f_c$ = 500Hz

The measured voltage over the load resistor due to the current of a photodiode in response to an LED at $f_c$: $V(t) = A(t)cos(2 \pi f_c t + \theta) $  

$A(t) = a + am(t) + n(t)$ where

a = the mean of the carrier amplitude, the voltage due to photocurrent, mV 

m(t) = the LED relative intensity fluctuation (common to all of the PDS), dimensionless, E[m] = 0

n(t) = the indepedent noise of each PD (shot, thermal, etc) within the band, mV

By using IQ Demodulation, A(t) can be recovered from the carrier wave.

The covariance of all of the A signal pairs from each diode pair can be found as 

$Cov(A_1, A_2) = a_1a_2Var(m)$ since n(t) is uncorrelated and all PDs see the same fractional fluctuation from the LED.  Thus, $Var(m)$ can be isolated as $Var(m) = \frac{Cov(A_1,A_2)}{a_1a_2}$.



Var(m) can be estimated by using the above formula and taking the mean of the most correlated pairs.

Since n(t) is uncorrelated, it can be found that $Var(n_i) = Var(A_i) - a_i^2Var(m)$ for each PD.

In [ ]:

#1. confirm vref, check if samples per cycle affects, confirm sine wave
#2. allow read from file for every block
#3. instead of checking between two PDs, sample multiple blocks from the same PD at different time instances and run algo
#4. test different bands for BP. 

#soures of error: reads are sequential from arduino circuit -> phase difference between all of the PDs.
if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)

NUM_SAMPLES_TO_SAVE = 30000   # set to None to run until interrupted

filtered_all = {s: [] for s in SENSORS}



sample_count = 0
sos_bp = signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos')
sos_zi = {s: signal.sosfilt_zi(sos_bp) for s in SENSORS}

if LOAD_FROM_FILE:
    reader = read_from_file_realtime(LOAD_FILE_PATH)
    NUM_SAMPLES_TO_SAVE = num_samples_in_file(LOAD_FILE_PATH)
try:
    while NUM_SAMPLES_TO_SAVE is None or sample_count < NUM_SAMPLES_TO_SAVE:
        if LOAD_FROM_FILE:
            data = reader()
        else:
            data = try_read_realtime(ser)
        if data is not None:
            for s in SENSORS:
                chunk_raw = adc_to_mv(data[s])  # convert to mV
                # bandpass this chunk, carrying filter state across chunks
                chunk_filt, sos_zi[s] = signal.sosfilt(sos_bp, chunk_raw, zi=sos_zi[s])
                filtered_all[s].append(chunk_filt)
            sample_count += SAMPLES_PER_PACKET
except KeyboardInterrupt:
    print("Stopping...")
finally:
    if ser is not None and ser.is_open:
        ser.close()

# skip 200 samples to avoid sos filter transient noise
filt = {s: np.concatenate(filtered_all[s])[200:] for s in SENSORS}
n = min(len(filt[s]) for s in SENSORS)
t = np.arange(n) / Fs   # time vector in samples, not packets

# A = demodulated message (mV)
# a = mean message amplitude
TRANSIENT = 150  # samples to drop after the FIR lowpass settles

A = {}
a = {}
for s in SENSORS:
    x = filt[s][:n]
    I = np.cos(2 * np.pi * CARRIER_FREQ * t) * x
    Q = -np.sin(2 * np.pi * CARRIER_FREQ * t) * x
    I_f, _ = fir_lowpass_filter(I, cutoff_hz=150, fs=Fs)
    Q_f, _ = fir_lowpass_filter(Q, cutoff_hz=150, fs=Fs)
    A[s] = (2 * np.sqrt(I_f**2 + Q_f**2))[TRANSIENT:]
    a[s] = np.mean(A[s])


X = np.vstack([A[s] for s in SENSORS])
cov_A = np.cov(X)         # mV^2
corr_A = np.corrcoef(X)   # dimensionless [-1, 1]
a_vec = np.array([a[s] for s in SENSORS])
norm_cov = cov_A / np.outer(a_vec, a_vec)   # cov/(a_i a_j); off-diag would be const if the same fluctuation is shared across all channels

def print_matrix(M, title, fmt="{:>12.4f}"):
    print(f"\n{title}")
    print(" " * 10 + "".join(f"{ch:>12}" for ch in SENSORS))
    for i, ch in enumerate(SENSORS):
        print(f"{ch:>10}" + "".join(fmt.format(M[i, j]) for j in range(len(SENSORS))))

print_matrix(cov_A,
    "[1] COVARIANCE (mV^2) - diagonal = total fluctuations of each channel, off-diagonal = fluctuation shared between two channels")
print_matrix(corr_A,
    "[2] CORRELATION (-1 to 1)-  near 1 = channels track togther (shared LED), 0 = independent")
print_matrix(norm_cov,
    "[3] NORMALIZED COVARIANCE - off-diagonals = var(m) per pair; all equal => one shared LED fluctuation", fmt="{:>12.4e}")

# var(m): LED fluctuation variance as a fraction of signal. Averaged ONLY over
#  pairs that are correlated
#  Var(A): total variance of each sensors's message (the diagonal of cov_A)
# % noise: independent noise as a fraction of that channel's total variance,
#  Var(n_i) = Var(A_i) - a_i^2 * var(m).  Only meaningful for sensors
#  that are coupled to the LED.
CORR_GATE = 0.5  # only average pairs that actually share the common mode
offdiag = ~np.eye(len(SENSORS), dtype=bool)
coupled = offdiag & (np.abs(corr_A) > CORR_GATE)
var_m = norm_cov[coupled].mean()
n_pairs = int(coupled.sum() // 2)
print(f"\nvar(m) (mean over {n_pairs} coupled pairs, |corr|>{CORR_GATE}) = {var_m:.4e}   RMS = {np.sqrt(var_m):.4f}")

print(f"\n{'channel':>10}{'Var(A) mV^2':>14}{'Var(n) mV^2':>14}{'% noise':>10}{'coupled':>9}")
for i, s in enumerate(SENSORS):
    var_total = cov_A[i, i]
    var_noise = var_total - a_vec[i]**2 * var_m
    pct_noise = 100 * var_noise / var_total
    is_coupled = bool(coupled[i].any())  # shares the common mode with at least one channel
    flag = "yes" if is_coupled else "NO"
    print(f"{s:>10}{var_total:>14.4f}{var_noise:>14.4f}{pct_noise:>9.1f}%{flag:>9}")
print("decoupled channel: a_i^2*var(m) assumption breaks, % noise unreliable")


## Variance of A Single Sensor over Sequential Time Periods

In [ ]:

#1. confirm vref, check if samples per cycle affects, confirm sine wave
#2. allow read from file for every block
#3. instead of checking between two PDs, sample multiple blocks from the same PD at different time instances and run algo
#4. test different bands for BP. 


if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)

LOAD_FILE_PATHS = [
    Path("VarianceData/06_30_2026/run_11_04_33/sample1.csv"),
    Path("VarianceData/06_30_2026/run_11_04_33/sample2.csv"),
    Path("VarianceData/06_30_2026/run_11_04_33/sample3.csv"),
    Path("VarianceData/06_30_2026/run_11_04_33/sample4.csv"),]

NUM_SAMPLES_TO_SAVE = 5000   # set to None to run until interrupted
SENSOR = SENSORS[0]
NUM_TRIALS = 6
PERIODS = [f"t{i+1}" for i in range(NUM_TRIALS)]
TRANSIENT = 200  # samples to drop to ensure filter settles
now = datetime.now()
savefolder_path = Path(f"temporal_variance_data/{now.strftime('%m_%d_%Y')}/run_{now.strftime('%H_%M_%S')}")

data = {}

#save 4 sets of data to file, then run this analysis on the saved file.  

if not LOAD_FROM_FILE:
    for i in range(NUM_TRIALS):
        savefile_path = savefolder_path / f"sample{i+1}.csv"
        save_data(NUM_SAMPLES_TO_SAVE, savefile_path, ser)
        rec = read_from_file_instant(savefile_path)      # one structured array (all sensor columns)
        data[i] = {s: rec[s] for s in SENSORS}      # split into one sample array per sensor
    if ser is not None and ser.is_open:
        ser.close()
else:
    for i in range(NUM_TRIALS):
        rec = read_from_file_instant(LOAD_FILE_PATHS[i])      # one structured array (all sensor columns)
        data[i] = {s: rec[s] for s in SENSORS}      # split into one sample array per sensor


# only look at SENSOR
data = {i: adc_to_mv(data[i][str(SENSOR)]) for i in range(NUM_TRIALS)}
times = {i: data[i]['timestamp'] for i in range(NUM_TRIALS)}

Fs = np.mean([1.0 / (np.median(np.diff(times[i])) * 1e-6) for i in range(NUM_TRIALS)])

sos_bp = signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos')
sos_lp = signal.butter(4, 150, btype='low', fs=Fs, output='sos')

filtered_data = {i: signal.sosfiltfilt(sos_bp, data[i]) for i in range(NUM_TRIALS)}
kept_filtered_data = {i: filtered_data[i][TRANSIENT:-TRANSIENT] for i in range(NUM_TRIALS)}


n = min(len(kept_filtered_data[i]) for i in range(NUM_TRIALS))
t = np.arange(n) / Fs 

A = {}
a = {}
for i in range(NUM_TRIALS):
    x = kept_filtered_data[i][:n]
    I = np.cos(2 * np.pi * CARRIER_FREQ * t) * x
    Q = -np.sin(2 * np.pi * CARRIER_FREQ * t) * x
    I_f = signal.sosfiltfilt(sos_lp, I)
    Q_f = signal.sosfiltfilt(sos_lp, Q)
    A[i] = (2 * np.sqrt(I_f**2 + Q_f**2))[TRANSIENT:-TRANSIENT]
    a[i] = np.mean(A[i])

a_vec = np.array([a[i] for i in range(NUM_TRIALS)])

var_A = np.array([np.var(A[i]) for i in range(NUM_TRIALS)])  # mV^2, total fluctuation per window
std_A = np.sqrt(var_A) # mV
frac  = std_A / a_vec # std/mean: fractional fluctuation (m + n combined)

result = f"A is the message encoded in the carrier wave (the total noise signal, LED FLUCTUATION + CIRCUIT NOISE + INTERFERENCE...)\n" 
result += f"a is the mean of the demodulated message\n"
result += f"Var(A) is the total variance of A\n"
result += f"std(A) is the standard deviation of A\n"
result += f"std/a is the fractional fluctuation (std/mean) for each time window.\n"

result += f"\nSame sensor ({SENSOR}) over {NUM_TRIALS} sequential time windows, Fs = {Fs:.2f} Hz\n"
result += f"{'window':>8}{'a (mV)':>12}{'Var(A) mV^2':>14}{'std(A) mV':>12}{'std/a':>12}\n"
for i in range(NUM_TRIALS):
    result += f"{PERIODS[i]:>8}{a_vec[i]:>12.4f}{var_A[i]:>14.4f}{std_A[i]:>12.4f}{frac[i]:>12.4e}\n"

# across-window summary: the spread of Var(A) shows how stationary the total
# fluctuation is. CV = std/mean of Var(A) across windows; small => stationary.
mean_var = var_A.mean()
std_var  = var_A.std(ddof=1)
cv = std_var / mean_var

result += f"\nVar(A) across windows: E[Var(A)] = {mean_var:.4f} mV^2, std = {std_var:.4f} mV^2, spread (CV) = {100*cv:.1f}%"
print(result)
with open(savefolder_path / "summary.txt", "w") as f:
    f.write(result + "\n")





## Find the Variance Between Sensors 2

In [20]:
if not LOAD_FROM_FILE:
    ser = serial.Serial(PORT, BAUD, timeout=1)

LOAD_FILE_PATHS = [
    Path("VarianceData/06_30_2026/run_11_04_33/sample1.csv"),
    Path("VarianceData/06_30_2026/run_11_04_33/sample2.csv"),
    Path("VarianceData/06_30_2026/run_11_04_33/sample3.csv"),
    Path("VarianceData/06_30_2026/run_11_04_33/sample4.csv"),]

NUM_SAMPLES_TO_SAVE = 10000   # set to None to run until interrupted
NUM_TRIALS = 2
CORR_GATE = 0.5  # only average pairs that actually share the common mode
PERIODS = [f"t{i+1}" for i in range(NUM_TRIALS)]
TRANSIENT = 200  # samples to drop on each end to ensure filter settles
now = datetime.now()
savefolder_path = Path(f"cross_sensor_variance_data/{now.strftime('%m_%d_%Y')}/run_{now.strftime('%H_%M_%S')}")

# data[trial] -> {sensor: 1-D array of that sensor's raw ADC samples for the trial}
data = {}

if not LOAD_FROM_FILE:
    for i in range(NUM_TRIALS):
        savefile_path = savefolder_path / f"sample{i+1}.csv"
        save_data(NUM_SAMPLES_TO_SAVE, savefile_path, ser)
        rec = read_from_file_instant(savefile_path)      
        data[i] = {s: rec[s] for s in SENSORS}   
    if ser is not None and ser.is_open:
        ser.close()
else:
    for i in range(NUM_TRIALS):
        rec = read_from_file_instant(LOAD_FILE_PATHS[i])
        data[i] = {s: rec[s] for s in SENSORS}


# filters:
# sos_bp: isolate the carrier (CARRIER_FREQ +/- 2 Hz)
# sos_lp: extract the demodulated envelope after mixing the carrier down to baseband 
# Both zero-phase (filtfilt)
sos_bp = signal.butter(4, [CARRIER_FREQ - 2, CARRIER_FREQ + 2], btype='bp', fs=Fs, output='sos')
sos_lp = signal.butter(4, 150, btype='low', fs=Fs, output='sos')

offdiag = ~np.eye(len(SENSORS), dtype=bool)   # mask that selects off-diagonal (cross) entries


def iq_demodulate(carrier_signal, t):
    I = signal.sosfiltfilt(sos_lp,  np.cos(2 * np.pi * CARRIER_FREQ * t) * carrier_signal)
    Q = signal.sosfiltfilt(sos_lp, -np.sin(2 * np.pi * CARRIER_FREQ * t) * carrier_signal)
    return 2.0 * np.sqrt(I ** 2 + Q ** 2)


result = "A_i is the demodulated message of sensor i (LED FLUCTUATION + CIRCUIT NOISE + INTERFERENCE...)\n"
result += "a_i is the mean carrier amplitude of sensor i\n"
result += "Cov(A_i, A_j) = a_i * a_j * Var(m)  ->  Var(m) = Cov(A_i, A_j) / (a_i * a_j)\n"
result += "Var(n_i) = Var(A_i) - a_i^2 * Var(m)  (independent per-diode noise)\n"
result += f"\nAcross {len(SENSORS)} sensors over {NUM_TRIALS} independent windows, Fs = {Fs:.2f} Hz\n"

var_m_trials = []

for trial in range(NUM_TRIALS):
    # raw ADC -> mV -> bandpass around the carrier, then drop the filter transient
    bandpassed = {}
    for s in SENSORS:
        v_mv = adc_to_mv(data[trial][s])
        v_bp = signal.sosfiltfilt(sos_bp, v_mv)
        bandpassed[s] = v_bp[TRANSIENT:-TRANSIENT]

    n = min(len(bandpassed[s]) for s in SENSORS)   # common length across channels
    t = np.arange(n) / Fs

    # recover each channel's envelope A_i(t) and its mean amplitude a_i
    # model:  A_i(t) = a_i + a_i*m(t) + n_i(t)
    A = {}
    a = {}
    for s in SENSORS:
        envelope = iq_demodulate(bandpassed[s][:n], t)
        A[s] = envelope[TRANSIENT:-TRANSIENT] # drop the low-pass edge transient
        a[s] = A[s].mean()

    X = np.vstack([A[s] for s in SENSORS])
    a_vec  = np.array([a[s] for s in SENSORS])
    cov_A  = np.cov(X) # mV^2, Var(A_i) on the diagonal
    corr_A = np.corrcoef(X)  # dimensionless, in [-1, 1]
    norm_cov = cov_A / np.outer(a_vec, a_vec) # Cov(A_i,A_j) / (a_i * a_j)

    # mask out entries that aren't correlated; var(m) = mean of what survives (nan if none)
    gated = np.where(offdiag & (corr_A > CORR_GATE), norm_cov, np.nan)
    var_m = np.nanmean(gated)
    n_pairs = int(np.isfinite(gated).sum() // 2)
    is_coupled = np.isfinite(gated).sum(axis=1) > 0
    var_m_trials.append(var_m)


    var_A = np.diag(cov_A)  # total envelope variance per channel
    var_common = a_vec ** 2 * var_m # portion explained by the shared LED
    var_n = var_A - var_common # remainder = independent, uncorrelated noise

    result += f"\n{'=' * 64}\nTRIAL {PERIODS[trial]}\n{'=' * 64}\n"
    result += fmt_matrix(cov_A, "[1] COVARIANCE (mV^2) - diagonal = total fluctuations of each channel, off-diagonal = fluctuation shared between two channels")
    result += fmt_matrix(corr_A, "[2] CORRELATION (-1 to 1)-  near 1 = channels track togther (shared LED), 0 = independent")
    result += fmt_matrix(norm_cov, "[3] NORMALIZED COVARIANCE - off-diagonals = var(m) estimate per pair; all equal => one shared LED fluctuation", fmt="{:>12.4e}")
    result += f"\nvar(m) = {var_m:.4e}   RMS(m) = sqrt(var(m)) = {np.sqrt(var_m):.4f}"
    result += f"   (mean over {n_pairs} coupled pair(s), corr > {CORR_GATE})\n"

    # per-channel noise split:  Var(A_i) = a_i^2*Var(m) [common] + Var(n_i) [independent]
    result += f"\n{'channel':>10}{'a_i (mV)':>10}{'Var(A_i)':>12}{'a^2*Var(m)':>12}{'Var(n_i)':>12}{'% indep':>9}{'coupled':>9}\n"
    clamped_any = False
    for i, s in enumerate(SENSORS):
        vn = var_n[i]
        if vn < 0: # unphysical: a_i^2*Var(m) > Var(A_i)
            vn = 0.0  # catastrophic cancellation -> report as ~0
            clamped_any = True
        pct_indep = 100 * vn / var_A[i]
        flag = "yes" if is_coupled[i] else "NO"
        result += (f"{s:>10}{a_vec[i]:>10.3f}{var_A[i]:>12.4f}" f"{var_common[i]:>12.4f}{vn:>12.4f}{pct_indep:>8.1f}%{flag:>9}\n")
    if clamped_any:
        result += "  Var(n_i)=0 (clamped): a_i^2*Var(m) >= Var(A_i), i.e. ~100% common mode; private noise is below this method's resolution.\n"
    result += "  Var(n_i) is only trustworthy for coupled channels; a decoupled channel breaks the a_i^2*Var(m) split.\n"


result += f"\nE[RMS(m)] over {NUM_TRIALS} trial(s) = {np.sqrt(np.nanmean(var_m_trials)):.4f}  (fractional LED intensity fluctuation)\n"

result += f"\n{'=' * 64}\nINTERPRETATION\n{'=' * 64}\n"
result += (
    "Model:  A_i(t) = a_i + a_i*m(t) + n_i(t)     (demodulated envelope of diode i)\n"
    "  a_i    = mean carrier amplitude, optical power reaching diode i\n"
    "  m(t)   = LED intensity fluctuation, SHARED by every diode (E[m]=0)\n"
    "  n_i(t) = independent per-diode noise (shot/thermal) left inside the band\n"
    "\n"
)


print(result)
savefolder_path.mkdir(parents=True, exist_ok=True)
with open(savefolder_path / "summary.txt", "w") as f:
    f.write(result + "\n")

Saved 10000 samples to cross_sensor_variance_data/07_02_2026/run_12_53_32/sample1.csv
Saved 10000 samples to cross_sensor_variance_data/07_02_2026/run_12_53_32/sample2.csv
A_i is the demodulated message of sensor i (LED FLUCTUATION + CIRCUIT NOISE + INTERFERENCE...)
a_i is the mean carrier amplitude of sensor i
Cov(A_i, A_j) = a_i * a_j * Var(m)  ->  Var(m) = Cov(A_i, A_j) / (a_i * a_j)
Var(n_i) = Var(A_i) - a_i^2 * Var(m)  (independent per-diode noise)

Across 2 sensors over 2 independent windows, Fs = 1499.25 Hz

TRIAL t1

[1] COVARIANCE (mV^2) - diagonal = total fluctuations of each channel, off-diagonal = fluctuation shared between two channels
               BPW34_1     BPW34_2
   BPW34_1      0.0156      0.0239
   BPW34_2      0.0239      0.0426

[2] CORRELATION (-1 to 1)-  near 1 = channels track togther (shared LED), 0 = independent
               BPW34_1     BPW34_2
   BPW34_1      1.0000      0.9284
   BPW34_2      0.9284      1.0000

[3] NORMALIZED COVARIANCE - off-diagonals